In [3]:
import gzip
import zipfile
import requests
import pandas as pd
from io import BytesIO
from pathlib import Path

# ---------- CONFIG ----------
DATE = "20140101"
OUTPUT = Path("data/news/raw")
OUTPUT.mkdir(parents=True, exist_ok=True)

GKG_URL = f"http://data.gdeltproject.org/gkg/{DATE}.gkg.csv.zip"

# Replace later with aliases loaded from your DB
KEYWORDS = [
    "TCS",
    "TATA CONSULTANCY",
    "INFOSYS",
    "RELIANCE",
    "HDFC BANK"
]
# ----------------------------

print("Downloading:", GKG_URL)

r = requests.get(GKG_URL, timeout=60)
r.raise_for_status()

z = zipfile.ZipFile(BytesIO(r.content))

filename = z.namelist()[0]

rows = []

with z.open(filename) as f:

    for line in f:

        try:
            line = line.decode("utf-8", errors="ignore")
        except:
            continue

        cols = line.strip().split("\t")

        if len(cols) < 5:
            continue

        record = line.upper()

        matched = False

        for k in KEYWORDS:
            if k in record:
                matched = True
                break

        if not matched:
            continue

        rows.append({
            "gkg_id": cols[0],
            "date": cols[1],
            "source": cols[-1]
        })

df = pd.DataFrame(rows)

df.drop_duplicates(inplace=True)

outfile = OUTPUT / f"{DATE}_urls.jsonl"

df.to_json(outfile,
           orient="records",
           lines=True,
           force_ascii=False)

print(df.head())

print(f"\nSaved {len(df)} records")

Downloading: http://data.gdeltproject.org/gkg/20140101.gkg.csv.zip
     gkg_id date                                             source
0  20140101    1  http://social.yourstory.com/2014/01/one-last-l...
1  20140101    1  http://timesofindia.indiatimes.com/india/aap-g...
2  20140101    1  http://www.rediff.com/business/slide-show/slid...
3  20140101    1  http://www.dnaindia.com/mumbai/report-aam-aadm...
4  20140101    1  http://www.spyghana.com/let-support-apostle-sa...

Saved 47 records


In [4]:
pip install newspaper4k lxml_html_clean beautifulsoup4 tqdm

Note: you may need to restart the kernel to use updated packages.


In [5]:
import json

count = 0

with open("data/news/raw/20140101_urls.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        count += len(row["source"].split("<UDIV>"))

print("Total URLs:", count)

Total URLs: 54
